# Carbon Mineralization in Construction Materials
## Multi-Agent LLM System (GPT-4o + Claude)

```
User Query
    |
Manager (GPT-4o)        -- interprets query, dispatches commands
    |
Orchestrator (Claude)   -- routes experts, coordinates coder, feedback loops
    |
+-- 7 Domain Experts (GPT-4o, parallel, RBAC-enforced tool access)
|     Chemistry      : kinetics-based phase evolution, SCM blends, pH
|     Thermodynamics : XRF -> initial hydration, phase evolution vs CO2,
|                      local equilibrium at depth, CSHQ decalcification
|                      (GEMS/cemdata18 backend, used only via tools)
|     Transport      : Fick's 2nd law, 6 diffusion models, carbonation depth
|     Micromechanics : 6-level Eshelby-Mori-Tanaka homogenization
|     Structural     : FEniCS FEM, 5 structures x 6 load conditions
|     MD             : LAMMPS nanoscale properties, 13-phase fallback DB
|     Engineering    : durability, service life, coupled E(x,t) profile
|
+-- Coder (Claude)      -- writes computation code, self-learning
+-- Reviewer (Claude)   -- vulnerability checks, PASS-only save
```

### Coupled Chemo-Transport-Mechanical Workflow

The engineering expert exposes `compute_coupled_chemo_transport_mechanical_profile`,
which chains the full physics pipeline:

1. **XRF + w/c -> GEMS full hydration** -> initial phase assemblage (CH, C-S-H, AFt,
   AFm, ...) and initial porosity.
2. **Transport solver** (Crank-Nicolson 1D FDM) -> CO2 concentration field c(x, t).
3. **Papadakis sharp-front dose** -> cumulative CO3^2- dose per depth per time.
4. **Per-depth GEMS local equilibrium** -> updated phase volumes, pH, porosity
   at each (x, t) (one GEMS engine per depth, monotone dose).
5. **6-level Eshelby-Mori-Tanaka** -> effective E, nu, K, G at each depth.
6. **Outputs:** E(x, t) durability map, carbonation depth vs sqrt(t), pH /
   porosity / phase profiles, K coefficient (mm/sqrt(yr)).

**Key rule:** GEMS is never invoked standalone. It is an internal backend of the
thermodynamics expert's tools and of the coupled-profile engineering tool.


In [ ]:
# Install dependencies
!pip install -q langchain langchain-openai langchain-anthropic langchain-core langgraph numpy scipy matplotlib

# Install FEniCS for Colab (optional; only needed for structural FEM)
try:
    import dolfin
    print("FEniCS already installed")
except ImportError:
    print("Installing FEniCS (~2 min)...")
    !wget -q "https://fem-on-colab.github.io/releases/fenics-install-release-real.sh" -O "/tmp/fenics-install.sh" && bash "/tmp/fenics-install.sh"
    try:
        import dolfin
        print("FEniCS installed")
    except ImportError:
        print("FEniCS install skipped (structural FEM disabled)")

print("All dependencies ready.")


In [ ]:
import os, sys

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = "/content/drive/MyDrive/1 Cement carbonation"
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(""))

sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

# Multi-agent system (primary entry point)
from colab_agent.orchestrator import CarbonMineralizationSystem

# Verification utilities (direct engine computations, for comparison)
from colab_agent.verification import run_verification, full_pipeline_visualization

# Coupled chemo-transport-mechanical driver
# (the engineering expert uses this internally; we also expose it here for
# direct inspection / plotting)
from colab_agent.coupled_carbonation import (
    compute_coupled_profile,
    plot_coupled_profile,
    plot_durability_time_evolution,
    run_coupled_reference,
)
from colab_agent.gems_engine import REFERENCE_XRF

print(f"Project root: {PROJECT_ROOT}")
print("System imported successfully.")
print(f"Reference XRF cements available: {list(REFERENCE_XRF.keys())}")


## API Keys


In [ ]:
import os

# OpenAI (for Manager + 7 Domain Experts)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OpenAI key loaded from Colab secrets")
except Exception:
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = input("Enter OpenAI API Key: ")
    print("OpenAI key set")

# Anthropic (for Orchestrator + Coder + Reviewer)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("CLAUDE_API_KEY")
    print("Anthropic key loaded from Colab secrets")
except Exception:
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = input("Enter Anthropic API Key: ")
    print("Anthropic key set")


In [ ]:
# Initialize multi-agent system
system = CarbonMineralizationSystem(
    workspace=os.path.join(PROJECT_ROOT, "_workspace"),
)
print("Multi-agent system ready.")
print(f"  Manager:      GPT-4o")
print(f"  Orchestrator: Claude")
print(f"  Experts:      7 domain agents (GPT-4o)")
print(f"    - Chemistry, Thermodynamics (GEMS/cemdata18 backend), Transport")
print(f"    - Micromechanics, Structural (FEM), MD, Engineering (coupled profile)")
print(f"  Coder:        Claude (self-learning)")
print(f"  Reviewer:     Claude (vulnerability check)")


---
## LLM Agent Analysis

The manager routes each query to the relevant experts. The thermodynamics
expert uses GEMS internally (XRF -> initial assemblage, phase evolution vs
CO2, local equilibrium at depth). The engineering expert exposes the full
coupled chemo-transport-mechanical profile.


In [ ]:
# Example 1 -- routine carbonation analysis (multiple experts in parallel)
result = system.analyze(
    "Analyze natural carbonation of OPC + 30% fly ash F concrete (w/c=0.45) "
    "for 50 years. Use the fib MC2010 diffusion model. "
    "Compute initial phase assemblage from GEMS, pore structure, carbonation "
    "depth, and effective mechanical properties using the 6-level homogenization "
    "at paste, mortar, and concrete levels."
)


In [ ]:
# Example 2 -- material comparison under accelerated carbonation
result = system.compare_materials(
    materials=["OPC", "OPC + 30% fly_ash_F", "OPC + 50% GGBS"],
    condition="accelerated carbonation at 5% CO2, w/c=0.50",
    duration_days=90,
)


In [ ]:
# Example 3 -- sensitivity analysis over RH
result = system.sensitivity_analysis(
    base_material="OPC",
    parameter="relative_humidity",
    values=[0.3, 0.5, 0.65, 0.8, 0.95],
)


In [ ]:
# Example 4 -- complex multi-physics workflow (manager dispatches to several experts)
result = system.analyze(
    "For an OPC + 20% GGBS concrete column (0.3 m x 1.0 m, w/c=0.45):\n"
    "1. Compute 50-year natural carbonation using the Papadakis diffusion model\n"
    "2. Get the carbonation depth and 6-level effective properties\n"
    "3. Run FEM analysis as a column under combined loading\n"
    "4. Assess service life per fib MC2010 carbonation limit (cover depth = 30 mm)"
)


---
## Coupled Chemo-Transport-Mechanical Profile

Run the full engineering pipeline via the LLM agent. The manager routes this
to the **engineering expert**, which calls
`compute_coupled_chemo_transport_mechanical_profile` internally.

Workflow:
`XRF -> GEMS hydration -> transport c(x,t) -> Papadakis dose -> local GEMS
equilibrium per depth -> 6-level Eshelby-MT -> E(x,t) durability map`.


In [ ]:
# Example 5 -- coupled profile via the LLM agent (engineering expert owns the tool)
result = system.analyze(
    "Using the reference OPC_CEM_I XRF composition (w/c=0.50, 25 C, "
    "atmospheric CO2 = 0.016 mol/m3), compute the full coupled chemo-transport-"
    "mechanical profile for 50 years with the fib MC2010 diffusion model. "
    "I want the E(x,t) durability map, the carbonation depth and K coefficient "
    "(mm/sqrt(yr)), and the surface pH / porosity evolution. Return a summary "
    "and interpret whether the K coefficient is physically realistic (expected "
    "range 2-5 mm/sqrt(yr) for natural carbonation)."
)


In [ ]:
# Example 6 -- direct call to compute_coupled_profile for plotting / inspection
# (same backend that the engineering expert uses; shown here so you can see the
#  raw arrays and plot both the 9-panel depth-focused map and the time-ordered
#  mid/long-term durability view)

xrf = REFERENCE_XRF["OPC_CEM_I"]

result = compute_coupled_profile(
    xrf_oxides=xrf,
    wc_ratio=0.50,
    alpha_hydration=1.0,
    temperature_C=25.0,
    C_env_CO2=0.016,            # atmospheric ~400 ppm CO2 at 25 C [mol/m3]
    diffusion_model="fib_mc2010",
    L_total=0.10,               # 100 mm depth
    nx=25,
    saturation=0.65,
    save_times_years=[0.5, 1, 2, 5, 10, 25, 50],
    level="paste",
)

print(f"K coefficient:    {result.meta['K_mm_per_sqrt_yr']:.2f} mm/sqrt(yr)")
print(f"Carbonation depth at 50 y: {result.carbonation_depth[-1]*1000:.1f} mm")
print(f"Surface pH  (t=50 y): {result.pH_field[-1, 0]:.2f}")
print(f"Core    pH  (t=50 y): {result.pH_field[-1, -1]:.2f}")
print(f"Surface E   (t=50 y): {result.E_field[-1, 0]/1e9:.2f} GPa")
print(f"Core    E   (t=50 y): {result.E_field[-1, -1]/1e9:.2f} GPa")

# (1) Depth-focused 9-panel snapshot: CO2, dose, pH, porosity, E profiles,
#     E(x,t) heatmap, carbonation depth vs sqrt(t), surface phase assemblage.
plot_coupled_profile(result, title="OPC CEM I -- natural carbonation, 50 y")

# (2) Time-ordered mid/long-term durability view:
#     (A) FEM-style 2D slab render of E(x, t) -- structural durability map.
#     (B) E vs time at fixed depths (cover / mid / core).
#     (C) CO2 vs time at fixed depths -- driving force history.
#     (D) Phase assemblage vs time at surface (solid) and core (dashed).
#     (E) Carbonation depth vs sqrt(t) with linear fit (K coefficient).
#     (F, G) Final depth profiles of E, CO2 and phase assemblage.
#     (H) Interpretation legend.
plot_durability_time_evolution(
    result,
    title="OPC CEM I -- mid/long-term durability (50 y)",
    depths_mm_track=[0.0, 5.0, 15.0, 30.0, 60.0],
)


---
## Verification: LLM Agent vs Direct Engine

Direct engine computations for comparison with LLM agent results.
Source is in `src/colab_agent/verification.py` to keep this notebook concise.


In [ ]:
# Run standard verification cases (3 cases with 6-panel plots each)
verification_results = run_verification()

# Or run a custom case:
# full_pipeline_visualization("OPC", scm_mix={"metakaolin": 0.10, "silica_fume": 0.05},
#                             diffusion_model="eshelby_mt", wc_ratio=0.40, duration_years=50)


---
## Export all figure data to CSV

Saves every array that drives the figures in this notebook into a
timestamped subfolder of `_workspace/exports_csv/<timestamp>/`. Two
sources are covered:

- **(A)** the `result` from Example 6 (direct `compute_coupled_profile` call)
- **(B)** every case in `verification_results` (Section "Direct verification")

For each `CoupledProfileResult` the following CSVs are written:

| File | Shape | Content |
|------|-------|---------|
| `grid.csv` | nx | depth index, depth [mm] |
| `times.csv` | nt | time index, time [yr], sqrt(t) [yr^0.5] |
| `co2_field.csv` | nt × nx | gas-phase CO2 [mol/m³] (rows = times) |
| `dissolved_dose_field.csv` | nt × nx | cumulative dissolved CO2 dose [mol/L paste] |
| `pH_field.csv` | nt × nx | pore-solution pH |
| `porosity_field.csv` | nt × nx | porosity |
| `E_field_GPa.csv` | nt × nx | effective Young's modulus [GPa] |
| `carbonation_degree_field.csv` | nt × nx | carbonation degree (0–1) |
| `phase_<name>.csv` | nt × nx | volume fraction of each phase |
| `carbonation_depth.csv` | nt | t, sqrt(t), x_c [mm] |
| `summary.csv` | scalars | K, D_eff, E0, surface/core E and pH, etc. |


In [ ]:
# ====================================================================
# Export every figure-driving array in this notebook to CSV.
#
# Run this cell AFTER you have executed
#   - Example 6  (creates `result`)
#   - the verification cell (creates `verification_results`)
# Either source is optional; whichever is present will be exported.
# All files land under  _workspace/exports_csv/<timestamp>/
# ====================================================================

import os
import csv
import datetime
import numpy as np

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
export_root = os.path.join(PROJECT_ROOT, "_workspace", "exports_csv", ts)
os.makedirs(export_root, exist_ok=True)
print(f"Exporting CSVs to:\n  {export_root}\n")


def _safe_name(s):
    return "".join(c if (c.isalnum() or c in "._-") else "_" for c in str(s))


def _save_2d(path, data, x_mm, t_yr):
    """Save an (nt, nx) field with depth header row and time index column."""
    arr = np.asarray(data, dtype=float)
    if arr.ndim != 2:
        return
    nt, nx = arr.shape
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["time_years \\ depth_mm"] + [f"{x:.4f}" for x in x_mm[:nx]])
        for i in range(nt):
            w.writerow([f"{t_yr[i]:.6g}"] + [f"{v:.6e}" for v in arr[i]])


def _save_columns(path, header, columns):
    """Save several aligned 1D columns side-by-side."""
    cols = [np.asarray(c) for c in columns]
    n = min(len(c) for c in cols)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        for i in range(n):
            row = []
            for c in cols:
                v = c[i]
                if isinstance(v, (int, np.integer)):
                    row.append(str(int(v)))
                elif isinstance(v, (float, np.floating)):
                    row.append(f"{float(v):.6e}")
                else:
                    row.append(str(v))
            w.writerow(row)


def _save_summary(path, kv):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["key", "value"])
        for k, v in kv.items():
            w.writerow([k, v])


def export_coupled_result(result, out_dir, label="case"):
    """Explode one CoupledProfileResult into a folder of CSVs."""
    os.makedirs(out_dir, exist_ok=True)

    x_mm = np.asarray(result.x_grid) * 1000.0
    t_yr = np.asarray(result.times_years)
    sqrt_t = np.sqrt(np.clip(t_yr, 0.0, None))

    # Axes
    _save_columns(
        os.path.join(out_dir, "grid.csv"),
        ["depth_index", "depth_mm"],
        [np.arange(len(x_mm)), x_mm],
    )
    _save_columns(
        os.path.join(out_dir, "times.csv"),
        ["time_index", "time_years", "sqrt_time_yr0.5"],
        [np.arange(len(t_yr)), t_yr, sqrt_t],
    )

    # 2D fields driving the figure panels
    _save_2d(os.path.join(out_dir, "co2_field.csv"),
             result.c_CO2, x_mm, t_yr)
    _save_2d(os.path.join(out_dir, "dissolved_dose_field.csv"),
             result.cumulative_CO2, x_mm, t_yr)
    _save_2d(os.path.join(out_dir, "pH_field.csv"),
             result.pH_field, x_mm, t_yr)
    _save_2d(os.path.join(out_dir, "porosity_field.csv"),
             result.porosity_field, x_mm, t_yr)
    _save_2d(os.path.join(out_dir, "E_field_GPa.csv"),
             np.asarray(result.E_field) / 1e9, x_mm, t_yr)
    if hasattr(result, "nu_field") and result.nu_field is not None:
        _save_2d(os.path.join(out_dir, "nu_field.csv"),
                 result.nu_field, x_mm, t_yr)
    if hasattr(result, "carb_degree_field") and result.carb_degree_field is not None:
        _save_2d(os.path.join(out_dir, "carbonation_degree_field.csv"),
                 result.carb_degree_field, x_mm, t_yr)

    # One CSV per phase
    for pname, pfield in (result.phase_fields or {}).items():
        _save_2d(
            os.path.join(out_dir, f"phase_{_safe_name(pname)}.csv"),
            pfield, x_mm, t_yr,
        )

    # Carbonation depth vs sqrt(t)
    xc_mm = np.asarray(result.carbonation_depth) * 1000.0
    _save_columns(
        os.path.join(out_dir, "carbonation_depth.csv"),
        ["time_years", "sqrt_time_yr0.5", "carbonation_depth_mm"],
        [t_yr, sqrt_t, xc_mm],
    )

    # Summary scalars
    meta = dict(getattr(result, "meta", {}) or {})
    summary = {
        "label": label,
        "n_times": int(len(t_yr)),
        "n_depths": int(len(x_mm)),
        "L_total_m": float(x_mm[-1] / 1000.0),
        "K_mm_per_sqrt_yr": float(meta.get("K_mm_per_sqrt_yr", float("nan"))),
        "D_eff_m2_per_s": float(meta.get("D_eff", float("nan"))),
        "diffusion_model": meta.get("diffusion_model", ""),
        "wc_ratio": meta.get("wc_ratio", ""),
        "level": meta.get("level", ""),
        "initial_E_GPa": float(result.initial_E) / 1e9,
        "E_surface_final_GPa": float(result.E_field[-1, 0]) / 1e9,
        "E_core_final_GPa": float(result.E_field[-1, -1]) / 1e9,
        "pH_surface_final": float(result.pH_field[-1, 0]),
        "pH_core_final": float(result.pH_field[-1, -1]),
        "porosity_surface_final": float(result.porosity_field[-1, 0]),
        "porosity_core_final": float(result.porosity_field[-1, -1]),
        "carbonation_depth_final_mm": float(xc_mm[-1]),
    }
    _save_summary(os.path.join(out_dir, "summary.csv"), summary)
    return summary


written = []

# ---- (A) Example 6 result -----------------------------------------
if "result" in dir() and hasattr(result, "x_grid"):
    out_a = os.path.join(export_root, "example6_OPC_CEM_I")
    s = export_coupled_result(result, out_a, label="OPC_CEM_I_Example6")
    written.append(out_a)
    print(f"  [A] Example 6  -> {os.path.relpath(out_a, export_root)}")
    print(f"      K = {s['K_mm_per_sqrt_yr']:.2f} mm/sqrt(yr),  "
          f"x_c(final) = {s['carbonation_depth_final_mm']:.1f} mm")
else:
    print("  [A] Example 6 skipped (variable `result` not found - "
          "run Example 6 first).")


# ---- (B) verification_results -------------------------------------
if "verification_results" in dir() and verification_results:
    for i, vr in enumerate(verification_results, 1):
        cr = vr.get("coupled_result")
        if cr is None:
            print(f"  [B{i}] {vr.get('label','case')} skipped (no coupled_result)")
            continue
        label = vr.get("label", f"case_{i}")
        out_b = os.path.join(
            export_root, f"verification_{i:02d}_{_safe_name(label)}"
        )
        s = export_coupled_result(cr, out_b, label=label)
        written.append(out_b)
        print(f"  [B{i}] {label}")
        print(f"        -> {os.path.relpath(out_b, export_root)}")
        print(f"        K = {s['K_mm_per_sqrt_yr']:.2f} mm/sqrt(yr),  "
              f"x_c(final) = {s['carbonation_depth_final_mm']:.1f} mm")
else:
    print("  [B] verification_results skipped "
          "(run the verification cell first).")


# ---- Top-level index ----------------------------------------------
index_path = os.path.join(export_root, "INDEX.csv")
with open(index_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["folder", "files"])
    for d in written:
        files = sorted(os.listdir(d))
        w.writerow([os.path.relpath(d, export_root), ";".join(files)])

print(f"\nDone. {len(written)} case folder(s) written.")
print(f"Index: {index_path}")
